# 财务数据批量同步（手动调度）

**用途**：前端零改动，通过本 notebook 手动调取 ams_trae 后端的单只同步接口，按 symbol 列表批量更新财务 / 估值 / 市值数据。

**关键设计**
1. **限速**：每一次 HTTP 请求之后随机停顿 1–3 秒（而非每个 symbol 只停一次），避免 gm 接口锁定——同一 symbol 的多张表之间也会留出间隔。
2. **当日去重**：一个 symbol 的全部目标表都返回 success / no_data 后，调 `POST /bds/sync/mark-synced` 把 `bds_symbol_info.sync_at` 写为当日；下次运行时 `EXCLUDE_SYNCED_TODAY=True` 直接跳过当日已跑完的标的。
3. **重跑安全**：不做批次断点记录，中断后直接重跑整批即可 —— 同步接口按增量起点（`MAX(date)+1`）推进，已同步的表返回 `no_data`，幂等安全。

**前置条件**
1. ams_trae 后端已启动：`conda run -n fast311 uvicorn server_fast.main:app --host 0.0.0.0 --port 8008`
2. gm 终端已登录启动（否则同步返回 `error` / `no_data`）

**运行顺序**：配置 → 选择 symbol 来源 → 执行批量同步 → 结果汇总。失败项用末尾的 Cell 重试。


## 接口与口径约定

- **同步接口**：`POST /api/v1/bds/sync/{table}?symbol=xxx`（单只单表）。
- **标记接口**：`POST /api/v1/bds/sync/mark-synced?symbol=xxx`（写 `bds_symbol_info.sync_at = 服务器本地当日`）。
- **表名 `table`**：`fund-balance` / `fund-income` / `fund-cashflow` / `finance-deriv` / `daily-valuation` / `daily-mktvalue`。
- **返回**：HTTP 200，body `status` 三态 —— `success`（有新数据）/ `no_data`（无新数据，或 gm 未启动）/ `error`（失败）。**必须按 body.status 判断**，不能只看 HTTP 状态码。
- **增量**：财报按 `rpt_date` 增量（带 pub_date 保留策略）；每日估值 / 市值按 `trade_date` 增量。**重复运行幂等安全**。
- **首次全量**：库中无该 symbol 数据时从 `Config.EARLIEST_START_DATE`（1901-01-01）起拉取，各数据源自会截断到个股上市日 / 最早可得财报（平安银行财报可追到 1989-12-31），单只耗时明显变长。
- **当日去重**：`sync_at` 是 symbol 级标记，语义为「该标的当日已跑完全部目标表」。`no_data` 也算跑过（避免 gm 未启动时死循环重跑），`error` 不标记。
- **报告期预判**（2026-10-13 晚）：三大报表 + 财务指标在调 gm 前，先比对库内该表该标的 `MAX(rpt_date)` 与 `bds_symbol_info.latest_rpt_date`；已追平则直接返回 `no_data`（`message`=「已是最新报告期，无需导入」），**不发 gm 请求**。任一方为空则照常请求。需要强刷（财报更正回补 / 快报转正式）时给请求加 `force=true`。每日估值 / 市值按 `trade_date` 增量，不参与预判。
- **重跑**：不做批次断点文件；中断后直接重跑，靠增量起点幂等跳过（已同步的表返回 `no_data`）。


In [11]:
# ===================== 配置 =====================
# 从公共包导入（config / api / sync / executor 全部 re-export）
from sync_runner import *

# 如需临时覆盖（不改库源码），在此重新赋值：
# API_BASE = 'http://127.0.0.1:8008/api/v1'
# TABLES = ['fund-balance', 'fund-income', 'fund-cashflow', 'finance-deriv',
#           'daily-valuation', 'daily-mktvalue']
# RESULT_CSV = 'sync_results.csv'

print('配置就绪 | 后端:', API_BASE)
print('待同步表:', TABLES)


配置就绪 | 后端: http://127.0.0.1:8008/api/v1
待同步表: ['fund-balance', 'fund-income', 'fund-cashflow', 'finance-deriv', 'daily-valuation', 'daily-mktvalue']


In [ ]:
# ===================== 选择 symbol 来源 =====================
# scope 取值：
#   'explicit' 手写清单（SYMBOLS_EXPLICIT）
#   'all'      全市场（bds_symbol_info 全部股票）
#   'industry' 指定行业（SCOPE_INDUSTRY，行业名可用 /bds/symbol-industries 查看）
#   'index'    指数成分（SCOPE_INDEX_CODE，如 SHSE.000300）
SCOPE = 'index'

SYMBOLS_EXPLICIT = [
    # 'SHSE.600519',   # 贵州茅台
    # 'SZSE.000001',   # 平安银行
]
SCOPE_INDUSTRY = ''        # scope=industry 时填，如 '银行'
SCOPE_INDEX_CODE = 'SHSE.000300'
MAX_SYMBOLS = 0            # >0 时只取前 N 只（试跑用，0=不限）

# True：剔除当日已完成同步的标的（依据 bds_symbol_info.sync_at，服务器本地当日）
EXCLUDE_SYNCED_TODAY = True

SYMBOLS = resolve_symbols(
    SCOPE,
    explicit=SYMBOLS_EXPLICIT,
    industry=SCOPE_INDUSTRY,
    index_code=SCOPE_INDEX_CODE,
    max_symbols=MAX_SYMBOLS,
    exclude_synced_today=EXCLUDE_SYNCED_TODAY,
)
print('预览:', SYMBOLS[:10], '...' if len(SYMBOLS) > 10 else '')
print('总任务数:', len(SYMBOLS) * len(TABLES), '（symbol 数 x 表数）')


In [ ]:
# ===================== 执行批量同步 =====================
# MARK_SYNCED_WHEN_DONE=True：一个 symbol 的全部表都非 error 后，写 sync_at=当日
MARK_SYNCED_WHEN_DONE = True

TASKS = [(s, t) for s in SYMBOLS for t in TABLES]
RESULTS = run_tasks(TASKS, mark_synced_when_done=MARK_SYNCED_WHEN_DONE)
summarize(RESULTS, RESULT_CSV)


In [ ]:
# ===================== 仅重试失败项 =====================
# 重跑上一次执行中 status=error 的 (symbol, table)
RETRY_NO_DATA = True   # True: 连同 no_data（可能是 gm 未启动）一并重试

res_df = pd.DataFrame(RESULTS)
if RETRY_NO_DATA:
    failed = res_df[res_df['status'].isin(['error', 'no_data'])]
else:
    failed = res_df[res_df['status'] == 'error']
retry_tasks = list(dict.fromkeys(zip(failed['symbol'], failed['table'])))
print('待重试任务数:', len(retry_tasks))

if retry_tasks:
    RESULTS = RESULTS + run_tasks(retry_tasks)
    summarize(RESULTS, RESULT_CSV)
else:
    print('无失败项。')
